# Deep State Space Models (DSSM) for Nonlinear Dynamics

**Sequence-to-Sequence Modeling with State Space Models**

This tutorial demonstrates how Deep State Space Models enable:
1. **Direct sequence modeling** using Linear Recurrent Units (LRU)
3. **Multi-step prediction** with stacked SSM layers

In [1]:

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import optax
from jaxtyping import Array, Float, PRNGKeyArray
from lru import LRU, StackedSSM
from tqdm import tqdm
from utils import (
    create_test_model,
    load_and_preprocess_data,
    ProgressPlotter,
    visualize_results,
)


In [ ]:


def create_dssm_model(
    key: PRNGKeyArray,
    d_model: int,
    d_vars: int,
    n_layers: int = 3,
    d_hidden: int | None = None,
    ssm_first_layer: bool = True,
    n_steps: int | None = None,
    dropout: float = 0.0,
    norm: str = "layer",
    activation: str = "half_glu1",
    prenorm: bool = True,
    r_min: float = 0.0,
    r_max: float = 1.0,
    max_phase: float = 6.28,
) -> StackedSSM:
    """Factory function to create initialized DSSM model."""

    model = StackedSSM(
        d_model=d_model,
        d_vars=d_vars,
        n_layers=n_layers,
        key=key,
        d_hidden=d_hidden,
        ssm_first_layer=ssm_first_layer,
        n_steps=n_steps,
        dropout=dropout,
        norm=norm,
        activation=activation,
        prenorm=prenorm,
        r_min=r_min,
        r_max=r_max,
        max_phase=max_phase,
    )

    return model

In [ ]:


@eqx.filter_jit
def dssm_loss(
    model: StackedSSM,
    batch: Float[Array, "B T W C"],
    key: PRNGKeyArray,
) -> Float[Array, ""]:
    """Loss function for DSSM sequence prediction."""

    def single_loss(trajectory, single_key):

        predictions = model(trajectory[0:1], key=single_key)

        loss = jnp.mean((predictions - trajectory) ** 2)

        return loss

    # Split keys for batch
    batch_keys = jax.random.split(key, batch.shape[0])

    # Apply to batch and average
    batch_losses = jax.vmap(single_loss)(batch, batch_keys)
    return jnp.mean(batch_losses)

In [ ]:


@eqx.filter_jit
def training_step(
    model: StackedSSM,
    optimizer,
    opt_state,
    batch: Float[Array, "B T W C"],
    key: PRNGKeyArray,
):
    """Single training step with gradient computation and parameter update."""

    @eqx.filter_value_and_grad
    def compute_loss(model):
        return dssm_loss(model, batch, key)

    loss_value, grads = compute_loss(model)
    updates, opt_state = optimizer.update(
        grads,
        opt_state,
        model,
    )
    model = eqx.apply_updates(model, updates)

    return model, opt_state, loss_value

In [ ]:


def train_dssm_model(
    model: LRU,
    train_dataloader,
    test_dataloader=None,
    n_epochs: int = 100,
    learning_rate: float = 5e-4,
    grad_clip_norm: float = 1.0,
    visualize_every_n_epochs: int = 10,
    save_plots: bool = True,
    n_steps_test: int | None = None,
):
    """Train the DSSM using the provided dataloader.

    Args:
        model: StackedSSM model to train
        train_dataloader: Training dataloader
        test_dataloader: Test dataloader for visualization
        n_epochs: Number of training epochs
        learning_rate: Peak learning rate for cosine schedule
        grad_clip_norm: threshold for gradient clipping
        visualize_every_n_epochs: How often to create visualization plots
        save_plots: Whether to save visualization plots during training
        n_steps_test: Number of steps to test during visualization

    Returns:
        Tuple of (trained_model, training_losses, plotter)
    """
    print("Training Deep State Space Model...")

    # Initialize progress plotter
    plotter = None
    if save_plots and test_dataloader is not None:
        plotter = ProgressPlotter(
            output_dir="tmp_dssm",
            model_name="DSSM",
            framerate=10,
        )

    # Calculate total training steps for proper scheduling
    batches_per_epoch = len(train_dataloader)
    total_steps = n_epochs * batches_per_epoch

    # Setup optimizer with cosine schedule and gradient clipping
    schedule = optax.cosine_onecycle_schedule(
        transition_steps=total_steps,
        peak_value=learning_rate,
    )
    optimizer = optax.chain(
        optax.clip_by_global_norm(grad_clip_norm),
        optax.adamw(schedule),
    )
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

    # Training loop with tqdm progress bar
    losses = np.full(n_epochs, np.nan)
    key = jax.random.PRNGKey(42)

    with tqdm(range(n_epochs), desc="Training DSSM") as pbar:
        for epoch in pbar:
            epoch_losses = []

            # Iterate through batches
            for batch in train_dataloader:
                key, subkey = jax.random.split(key)
                model, opt_state, loss_value = training_step(
                    model,
                    optimizer,
                    opt_state,
                    batch,
                    subkey,
                )
                epoch_losses.append(loss_value)
                # break  # For debugging, remove this line for full training

            # Record average loss for this epoch
            avg_loss = jnp.mean(jnp.array(epoch_losses))
            losses[epoch] = avg_loss

            # Update progress bar
            pbar.set_postfix({"Loss": f"{avg_loss:.6f}"})

            # Visualization during training
            if plotter is not None and (epoch + 1) % visualize_every_n_epochs == 0:
                # Create test model with current trained weights for visualization
                current_test_model = (
                    create_test_model(model, n_steps_test)
                    if n_steps_test is not None
                    else model
                )
                plotter(
                    current_test_model,
                    test_dataloader,
                    losses,
                    show_plot=False,
                    show_loss_plot=False,
                )

    return model, losses, plotter

In [ ]:

# Training hyperparameters
n_epochs = 5000
n_steps_train = 100
n_steps_test = 200
batch_size = 25
learning_rate = 1e-3
grad_clip_norm = 1

# Model hyperparameters
n_layers = 2
d_hidden = 128
dropout = 0.0
activation = "gelu"

# Load data using shared pipeline
(
    train_dataloader,
    val_dataloader,
    test_dataloader,
) = load_and_preprocess_data(
    "data/string_nonlin_100_Gaussian_16000Hz_1.0s.npy",
    batch_size=batch_size,
    n_steps_train=n_steps_train,
    n_steps_test=n_steps_test,
    split=[0.8, 0.1, 0.1],
)

# Get data dimensions from first batch
train_sample_batch = next(iter(train_dataloader))
test_sample_batch = next(iter(test_dataloader))
B_train, T_train, W_train, C_train = train_sample_batch.shape
B_test, T_test, W_test, C_test = test_sample_batch.shape
print(f"Data dimensions: B={B_train}, T={T_train}, W={W_train}, C={C_train}")

# Create model
key = jax.random.PRNGKey(42)
model = create_dssm_model(
    key=key,
    d_model=W_train,
    d_vars=C_train,
    n_layers=n_layers,
    d_hidden=d_hidden,
    n_steps=T_train,
    dropout=dropout,
    activation=activation,
    ssm_first_layer=True,
    r_min=0.9,
    r_max=0.99,
    max_phase=np.pi / 2,
)

In [ ]:
param_count = sum(
    x.size
    for x in jax.tree.leaves(
        eqx.filter(model, eqx.is_array),
    )
)
print(f"Model has {param_count:,} parameters")

In [ ]:
trained_model, losses, plotter = train_dssm_model(
    model,
    train_dataloader,
    test_dataloader,
    n_epochs=n_epochs,
    learning_rate=learning_rate,
    grad_clip_norm=grad_clip_norm,
    n_steps_test=n_steps_test,
    save_plots=False,
    visualize_every_n_epochs=50,
)

In [ ]:

final_trained_model = eqx.tree_at(
    lambda m: m.first_layer.n_steps,
    trained_model,
    T_test,
)

def model_wrapper(x):
    return final_trained_model(x, key=jax.random.PRNGKey(0))

visualize_results(
    model_wrapper,
    test_dataloader,
    losses,
    model_name="DSSM",
    save_path=None,
    show_plot=True,
    show_loss_plot=False,
    n_steps_train=T_train,
)

print("Training completed!")

# Generate animation from training frames
if plotter is not None:
    plotter.render_animation("dssm_training.webm")


Things to try:
- Adjust number of layers and hidden dimensions
- Experiment with different activation functions (gelu, mlp, full_glu)
- Try different LRU parameters (r_min, r_max, max_phase)
- Compare with and without the first SSM layer